<a href="https://colab.research.google.com/github/Cumikkk/uts-visi-komputer-segmentasi-daun-tomat/blob/main/UTS_VisiKomputer_SegmentasiDaunTomat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import semua library yang dibutuhkan
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

print("Library berhasil diimport.")

In [ ]:
import os

# Sesuaikan nama file dengan yang sudah diupload ke Colab
IMAGE_PATHS = {
    "Daun Sehat"    : "/content/sehat.webp",
    "Septoria Leaf" : "/content/septoria.webp",
    "Early Blight"  : "/content/early_blight.jpeg",
}

# Load semua gambar dari folder lokal
images_bgr = {}
for label, path in IMAGE_PATHS.items():
    if os.path.exists(path):
        img = cv2.imread(path)
        if img is not None:
            images_bgr[label] = img
            print(f"[OK] {label} — ukuran: {img.shape[1]}x{img.shape[0]} px")
        else:
            print(f"[GAGAL] {label} — file tidak bisa dibaca (coba konversi ke .jpg dulu)")
    else:
        print(f"[TIDAK DITEMUKAN] {label} — cek nama file di /content/")

In [ ]:
# Ukuran target resize semua gambar
TARGET_SIZE = (300, 300)

def preprocess(img_bgr):
    # Resize ke ukuran seragam
    resized = cv2.resize(img_bgr, TARGET_SIZE)
    # Gaussian Blur untuk mengurangi noise sebelum segmentasi
    blurred = cv2.GaussianBlur(resized, (5, 5), 0)
    # Konversi BGR ke HSV — lebih robust untuk deteksi warna daun
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
    return resized, blurred, hsv

# Jalankan preprocessing untuk semua gambar dan simpan hasilnya
preprocessed = {}
for label, img_bgr in images_bgr.items():
    resized, blurred, hsv = preprocess(img_bgr)
    preprocessed[label] = (resized, blurred, hsv)
    print(f"[OK] Preprocessing selesai: {label}")

In [ ]:
# HSV hijau — range diperluas untuk berbagai kondisi pencahayaan
HSV_GREEN_LOW  = np.array([20,  20,  20])
HSV_GREEN_HIGH = np.array([95, 255, 255])

# HSV kuning-coklat — area penyakit dan daun menguning
HSV_YELLOW_LOW  = np.array([8,  20,  40])
HSV_YELLOW_HIGH = np.array([28, 255, 255])

# HSV abu-abu kehijauan — khusus untuk Early Blight yang daunnya gelap kebiruan
HSV_GRAY_GREEN_LOW  = np.array([85,  10,  40])
HSV_GRAY_GREEN_HIGH = np.array([140, 80, 180])

# Kernel morfologi ellipse 5x5 sesuai paper
KERNEL    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
# Kernel lebih besar untuk closing — menutup lubang lebih agresif
KERNEL_LG = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))

print("Parameter segmentasi siap.")
print(f"HSV Hijau      : {HSV_GREEN_LOW} s/d {HSV_GREEN_HIGH}")
print(f"HSV Kuning     : {HSV_YELLOW_LOW} s/d {HSV_YELLOW_HIGH}")
print(f"HSV Abu-Hijau  : {HSV_GRAY_GREEN_LOW} s/d {HSV_GRAY_GREEN_HIGH}")
print(f"Kernel Normal  : Ellipse 5x5")
print(f"Kernel Besar   : Ellipse 11x11")

In [ ]:
def grabcut_roi(img_bgr):
    """
    GrabCut untuk estimasi awal foreground (area daun).
    Margin dinamis: 8% sisi, tapi rect minimum 10px dari tepi.
    """
    mask_gc = np.zeros(img_bgr.shape[:2], np.uint8)
    h, w    = img_bgr.shape[:2]
    margin  = int(min(h, w) * 0.08)
    margin  = max(margin, 10)
    rect    = (margin, margin, w - margin * 2, h - margin * 2)

    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    cv2.grabCut(img_bgr, mask_gc, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)

    fg_mask = np.where(
        (mask_gc == cv2.GC_FGD) | (mask_gc == cv2.GC_PR_FGD), 255, 0
    ).astype(np.uint8)
    return fg_mask

def remove_bright_background(img_hsv):
    """
    Buat mask untuk EXCLUDE background putih/terang.
    Piksel dengan Saturation sangat rendah + Value sangat tinggi
    = background putih → dibuang dari mask.
    Mengatasi masalah Baris 1 (background putih ikut tertangkap).
    """
    # Background putih: saturation rendah (<30) dan value tinggi (>200)
    mask_bright = cv2.inRange(img_hsv, np.array([0, 0, 200]), np.array([180, 30, 255]))
    # Inverse: area yang BUKAN background putih
    mask_not_bright = cv2.bitwise_not(mask_bright)
    return mask_not_bright

def remove_round_objects(mask, img_resized, min_aspect=0.5, max_aspect=2.0, min_solidity=0.80):
    """
    Filter kontur berdasarkan bentuk untuk membuang objek bulat (tomat).
    Daun → aspect ratio tidak terlalu bulat, solidity tidak terlalu tinggi.
    Tomat → sangat bulat → solidity tinggi, aspect ratio mendekati 1.
    Mengatasi masalah Baris 3 (tomat hijau ikut tersegmentasi).
    """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask_filtered = np.zeros_like(mask)

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 500:
            continue

        # Hitung solidity: area / area convex hull
        hull    = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        solidity  = area / hull_area if hull_area > 0 else 0

        # Hitung aspect ratio dari bounding rect
        x, y, w, h = cv2.boundingRect(cnt)
        aspect = float(w) / h if h > 0 else 0

        # Tomat: solidity tinggi (>0.85) DAN aspect ratio mendekati 1 (0.75–1.25)
        is_round = (solidity > max_solidity) and (0.75 < aspect < 1.25)
        if not is_round:
            cv2.drawContours(mask_filtered, [cnt], -1, 255, thickness=cv2.FILLED)

    return mask_filtered

# Threshold solidity untuk filter objek bulat
max_solidity = 0.88

def segment_leaf(img_hsv, img_resized):
    """
    Segmentasi daun mengikuti metode paper + perbaikan:
      Step 1 : GrabCut → estimasi ROI foreground awal
      Step 2 : Exclude background putih/terang
      Step 3 : Color Thresholding HSV (hijau + kuning + abu-hijau)
      Step 4 : Gabungkan mask GrabCut × HSV × exclude-bright
      Step 5 : Morfologi Opening → buang noise kecil
      Step 6 : Morfologi Closing → tutup lubang dalam daun
      Step 7 : Filter bentuk bulat → buang tomat
      Step 8 : Ambil kontur terbesar → objek daun utama
      Step 9 : Bitwise Masking → terapkan ke gambar asli
    """
    # Step 1: GrabCut pre-mask
    gc_mask = grabcut_roi(img_resized)

    # Step 2: Mask exclude background putih
    mask_not_bright = remove_bright_background(img_hsv)

    # Step 3a: Mask warna hijau
    mask_green    = cv2.inRange(img_hsv, HSV_GREEN_LOW,      HSV_GREEN_HIGH)
    # Step 3b: Mask warna kuning-coklat (area penyakit)
    mask_yellow   = cv2.inRange(img_hsv, HSV_YELLOW_LOW,     HSV_YELLOW_HIGH)
    # Step 3c: Mask abu-hijau (daun Early Blight yang kebiruan gelap)
    mask_gray_grn = cv2.inRange(img_hsv, HSV_GRAY_GREEN_LOW, HSV_GRAY_GREEN_HIGH)

    # Step 3d: Gabungkan semua mask HSV
    mask_hsv = cv2.bitwise_or(mask_green,  mask_yellow)
    mask_hsv = cv2.bitwise_or(mask_hsv,    mask_gray_grn)

    # Step 4: Kombinasi GrabCut + HSV + exclude background putih
    mask_combined = cv2.bitwise_and(mask_hsv,      gc_mask)
    mask_combined = cv2.bitwise_and(mask_combined, mask_not_bright)

    # Fallback: jika hasil terlalu kosong (<5%), pakai HSV + exclude bright saja
    coverage = np.count_nonzero(mask_combined) / (mask_combined.shape[0] * mask_combined.shape[1])
    if coverage < 0.05:
        mask_combined = cv2.bitwise_and(mask_hsv, mask_not_bright)

    # Step 5: Opening — buang noise kecil di luar daun
    mask_opened = cv2.morphologyEx(mask_combined, cv2.MORPH_OPEN,  KERNEL,    iterations=2)

    # Step 6: Closing — tutup lubang di dalam daun lebih agresif
    mask_closed = cv2.morphologyEx(mask_opened,   cv2.MORPH_CLOSE, KERNEL_LG, iterations=3)

    # Step 7: Filter objek bulat (buang tomat hijau)
    mask_filtered = remove_round_objects(mask_closed, img_resized)

    # Step 8: Ambil kontur terbesar sebagai objek daun utama
    contours, _ = cv2.findContours(mask_filtered, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask_contour = np.zeros_like(mask_filtered)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        cv2.drawContours(mask_contour, [largest], -1, 255, thickness=cv2.FILLED)

    # Step 9: Bitwise Masking — terapkan mask final ke gambar asli
    img_segmented = cv2.bitwise_and(img_resized, img_resized, mask=mask_contour)

    return mask_combined, mask_closed, mask_contour, img_segmented

print("Fungsi segmentasi diperbarui (filter bright background + filter bulat) siap.")

In [ ]:
# Proses segmentasi untuk semua gambar
segmented_results = {}
for label, (img_resized, img_blurred, img_hsv) in preprocessed.items():
    mask_combined, mask_closed, mask_contour, img_segmented = segment_leaf(img_hsv, img_resized)
    segmented_results[label] = (mask_combined, mask_closed, mask_contour, img_segmented)
    print(f"[OK] Segmentasi selesai: {label}")

In [ ]:
fig, axes = plt.subplots(
    nrows=len(images_bgr),
    ncols=5,
    figsize=(20, 5 * len(images_bgr))
)

fig.suptitle(
    "Segmentasi Citra Daun Tomat\n"
    "Metode: Color Thresholding (HSV) + Morfologi (Opening & Closing) + Contour Filtering\n"
    "Referensi: Azli et al., Jurnal Informatika Poltekharber, 2025",
    fontsize=13, fontweight='bold', y=1.01
)

col_titles = [
    "1. Citra Asli",
    "2. Channel Hue (HSV)",
    "3. Mask Gabungan\n(Sebelum Morfologi)",
    "4. Mask Setelah\nMorfologi (Open+Close)",
    "5. Hasil Segmentasi\n(Bitwise Masking)"
]

for col_idx, title in enumerate(col_titles):
    axes[0][col_idx].set_title(title, fontsize=10, fontweight='bold')

for row_idx, label in enumerate(images_bgr.keys()):
    img_resized, _, img_hsv            = preprocessed[label]
    mask_combined, mask_closed, mask_contour, img_segmented = segmented_results[label]

    img_rgb     = cv2.cvtColor(img_resized,   cv2.COLOR_BGR2RGB)
    img_seg_rgb = cv2.cvtColor(img_segmented, cv2.COLOR_BGR2RGB)
    h_channel   = img_hsv[:, :, 0]

    row_data = [img_rgb, h_channel, mask_combined, mask_closed, img_seg_rgb]
    cmaps    = [None,    'hsv',     'gray',        'gray',      None]

    for col_idx, (data, cmap) in enumerate(zip(row_data, cmaps)):
        ax = axes[row_idx][col_idx]
        ax.imshow(data, cmap=cmap)
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(label, fontsize=11, fontweight='bold',
                          rotation=90, labelpad=10, va='center')

plt.tight_layout()
plt.savefig("hasil_segmentasi_daun_tomat.png", dpi=120, bbox_inches='tight')
plt.show()
print("Visualisasi disimpan: hasil_segmentasi_daun_tomat.png")

In [ ]:
print("="*60)
print("ANALISIS KUANTITATIF HASIL SEGMENTASI")
print("="*60)
print(f"{'Label':<20} {'Area Mask (px)':<18} {'Total Pixel':<15} {'Coverage (%)'}")
print("-"*60)

for label in images_bgr.keys():
    _, _, mask_contour, _ = segmented_results[label]
    area_mask   = np.count_nonzero(mask_contour)
    total_pixel = mask_contour.shape[0] * mask_contour.shape[1]
    coverage    = (area_mask / total_pixel) * 100
    print(f"{label:<20} {area_mask:<18} {total_pixel:<15} {coverage:.2f}%")

print("="*60)
print("\nKeterangan:")
print("  Area Mask    : jumlah piksel yang terdeteksi sebagai daun")
print("  Total Pixel  : total piksel gambar (300x300 = 90.000)")
print("  Coverage (%) : persentase area daun terhadap seluruh gambar")

In [ ]:
# Gunakan Early Blight sebagai contoh detail
sample_label = "Early Blight"
img_resized, _, img_hsv = preprocessed[sample_label]

# Pakai GrabCut + exclude bright (sama persis dengan segment_leaf)
gc_mask         = grabcut_roi(img_resized)
mask_not_bright = remove_bright_background(img_hsv)

mask_green    = cv2.inRange(img_hsv, HSV_GREEN_LOW,      HSV_GREEN_HIGH)
mask_yellow   = cv2.inRange(img_hsv, HSV_YELLOW_LOW,     HSV_YELLOW_HIGH)
mask_gray_grn = cv2.inRange(img_hsv, HSV_GRAY_GREEN_LOW, HSV_GRAY_GREEN_HIGH)
mask_hsv      = cv2.bitwise_or(mask_green, mask_yellow)
mask_hsv      = cv2.bitwise_or(mask_hsv,   mask_gray_grn)

mask_raw  = cv2.bitwise_and(mask_hsv, gc_mask)
mask_raw  = cv2.bitwise_and(mask_raw,  mask_not_bright)

# Fallback jika terlalu kosong
if np.count_nonzero(mask_raw) / (mask_raw.shape[0] * mask_raw.shape[1]) < 0.05:
    mask_raw = cv2.bitwise_and(mask_hsv, mask_not_bright)

mask_open  = cv2.morphologyEx(mask_raw,  cv2.MORPH_OPEN,  KERNEL,    iterations=2)
mask_close = cv2.morphologyEx(mask_open, cv2.MORPH_CLOSE, KERNEL_LG, iterations=3)

fig2, axes2 = plt.subplots(1, 4, figsize=(18, 5))
fig2.suptitle(
    f"Detail Tahapan Operasi Morfologi — Gambar: {sample_label}",
    fontsize=12, fontweight='bold'
)

steps = [
    (cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB), "Citra Asli",                          None),
    (mask_raw,                                      "Mask Sebelum Morfologi\n(GrabCut+HSV)", 'gray'),
    (mask_open,                                     "Setelah Opening\n(Buang Noise Luar)",   'gray'),
    (mask_close,                                    "Setelah Closing\n(Tutup Lubang Dalam)", 'gray'),
]

for ax, (data, title, cmap) in zip(axes2, steps):
    ax.imshow(data, cmap=cmap)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig("detail_morfologi.png", dpi=120, bbox_inches='tight')
plt.show()
print("Detail morfologi disimpan: detail_morfologi.png")